# Unit 7: 实战项目 — CIFAR-10 图像分类完整演练

## 项目概述

本项目综合运用前 6 个单元的知识，完成一个**完整的图像分类项目**。

**目标**：在 CIFAR-10 数据集上训练一个高精度图像分类器。

**任务**：10 类分类（airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck）

**流程**：
1. 数据探索与预处理
2. 基线模型 (Baseline)
3. 改进模型 (ResNet 风格)
4. 迁移学习方案
5. 超参数调优实验
6. 综合评估（混淆矩阵、各类准确率）
7. 错误分析
8. 模型导出与推理

## 7.1 环境准备与数据加载

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time
import copy

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

DATA_DIR = Path("./data")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 128
NUM_WORKERS = 0

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

full_train_dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=train_transform) # 50000
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform) # 10000

train_size = 45000
val_size = 5000
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {train_size:,} | Val: {val_size:,} | Test: {len(test_dataset):,}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


## 7.2 数据探索

CIFAR-10 的设计就是每个类别恰好包含 5,000 张训练图像（以及 1,000 张测试图像）  
5000 * 10 = 50,000 张训练图像  
1000 * 10 = 10,000 张测试图像

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
from matplotlib.gridspec import GridSpec

# 1. 创建 figure
fig = plt.figure(figsize=(14, 5), constrained_layout=True)

# 2. 使用 GridSpec 划分：左侧 60%，右侧 40%
gs = GridSpec(1, 2, width_ratios=[0.5, 0.5], figure=fig)

# 3. 左侧：柱状图（占 60%）
ax1 = fig.add_subplot(gs[0])
class_counts = [0] * 10
for _, label in raw_dataset:
    class_counts[label] += 1

colors = plt.cm.tab10(np.arange(10))
ax1.bar(CIFAR10_CLASSES, class_counts, color=colors)
ax1.set_title("Class Distribution (Training Set)")
ax1.set_ylabel("Number of Samples")
ax1.tick_params(axis="x", rotation=45)
for i, v in enumerate(class_counts):
    ax1.text(i, v + 50, str(v), ha="center", fontsize=8)

# 4. 右侧：ImageGrid（占 40%）
# 注意：这里用 gs[1] 作为父容器区域
grid = ImageGrid(
    fig,
    gs[1],                # ← 关键！绑定到 GridSpec 的第二个区域
    nrows_ncols=(2, 5),
    axes_pad=0.3,
    share_all=True,
)

# 5. 绘制 10 张示例图
for i, ax_inset in enumerate(grid):
    idx = np.where(np.array(raw_dataset.targets) == i)[0][0]
    img, _ = raw_dataset[idx]
    ax_inset.imshow(img)
    ax_inset.set_title(CIFAR10_CLASSES[i], fontsize=8)
    ax_inset.axis("off")


plt.show()

## 7.3 通用训练框架

定义一个可复用的训练引擎，贯穿整个项目。

In [ ]:
class TrainingEngine:
    def __init__(self, model, device, criterion=None, optimizer=None, scheduler=None):
        self.model = model.to(device)
        self.device = device
        self.criterion = criterion or nn.CrossEntropyLoss()
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
        self.best_val_acc = 0.0
        self.best_model_state = None

    def train_epoch(self, loader):
        self.model.train()
        total_loss, correct, total = 0, 0, 0
        pbar = tqdm(loader, desc="Training", leave=False)
        for data, target in pbar:
            data, target = data.to(self.device), target.to(self.device)
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.criterion(output, target)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += data.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        return total_loss / total, correct / total

    @torch.no_grad()
    def evaluate(self, loader, desc="Evaluating"):
        self.model.eval()
        total_loss, correct, total = 0, 0, 0
        all_preds, all_labels = [], []
        for data, target in tqdm(loader, desc=desc, leave=False):
            data, target = data.to(self.device), target.to(self.device)
            output = self.model(data)
            loss = self.criterion(output, target)
            total_loss += loss.item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += data.size(0)
            all_preds.append(pred.cpu())
            all_labels.append(target.cpu())
        all_preds = torch.cat(all_preds)
        all_labels = torch.cat(all_labels)
        return total_loss / total, correct / total, all_preds, all_labels

    def fit(self, train_loader, val_loader, epochs, save_path=None):
        for epoch in range(epochs):
            epoch_start = time.time()
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc, _, _ = self.evaluate(val_loader, desc="Validating")

            if self.scheduler:
                self.scheduler.step()

            self.history["train_loss"].append(train_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)

            lr = self.optimizer.param_groups[0]["lr"]
            elapsed = time.time() - epoch_start
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | "
                  f"LR: {lr:.2e} | Time: {elapsed:.1f}s")

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc
                self.best_model_state = copy.deepcopy(self.model.state_dict())
                if save_path:
                    self.save(save_path)
                    print(f"  >>> Best model saved (Val Acc: {val_acc:.4f})")

        self.model.load_state_dict(self.best_model_state)
        print(f"\nTraining complete. Best Val Acc: {self.best_val_acc:.4f}")

    def save(self, path):
        torch.save({
            "model_state_dict": self.best_model_state,
            "optimizer_state_dict": self.optimizer.state_dict(),
            "best_val_acc": self.best_val_acc,
            "history": self.history,
        }, path)

    def load(self, path, load_optimizer=False):
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        if load_optimizer and self.optimizer:
            self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.best_val_acc = checkpoint["best_val_acc"]
        self.history = checkpoint["history"]
        self.best_model_state = checkpoint["model_state_dict"]

    def predict(self, loader):
        _, _, preds, labels = self.evaluate(loader, desc="Predicting")
        return preds, labels

## 7.4 模型 1：基线 CNN

简单三层 CNN，作为性能基准。

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model_baseline = BaselineCNN()
print(f"BaselineCNN params: {sum(p.numel() for p in model_baseline.parameters()):,}")

In [ ]:
engine_baseline = TrainingEngine(
    model=BaselineCNN(),
    device=device,
)

engine_baseline.optimizer = optim.SGD(engine_baseline.model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
engine_baseline.fit(train_loader, val_loader, epochs=30,
                    save_path=str(CHECKPOINT_DIR / "baseline_cnn.pt"))

## 7.5 模型 2：ResNet 风格 CNN

综合 BatchNorm + 残差连接 + Dropout 的改进模型。

In [ ]:
class ResidualBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNetStyle(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 32

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )

        self.layer1 = self._make_layer(32, 3, stride=1)
        self.layer2 = self._make_layer(64, 3, stride=2)
        self.layer3 = self._make_layer(128, 3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

        self._initialize_weights()

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(ResidualBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model_resnet = ResNetStyle()
x = torch.randn(2, 3, 32, 32)
with torch.no_grad():
    print(f"Input {x.shape} -> Output {model_resnet(x).shape}")
print(f"ResNetStyle params: {sum(p.numel() for p in model_resnet.parameters()):,}")

In [ ]:
engine_resnet = TrainingEngine(
    model=ResNetStyle(),
    device=device,
)

engine_resnet.optimizer = optim.SGD(engine_resnet.model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
engine_resnet.scheduler = optim.lr_scheduler.CosineAnnealingLR(engine_resnet.optimizer, T_max=50)

engine_resnet.fit(train_loader, val_loader, epochs=50,
                  save_path=str(CHECKPOINT_DIR / "resnet_style.pt"))

## 7.6 训练过程可视化

In [ ]:
def plot_training_history(engines, labels, title="Training Comparison"):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for engine, label in zip(engines, labels):
        h = engine.history
        axes[0].plot(h["val_loss"], label=f"{label} (best: {engine.best_val_acc:.2%})")
        axes[1].plot(h["val_acc"], label=f"{label} (best: {engine.best_val_acc:.2%})")

    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Validation Loss")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Validation Accuracy")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_training_history([engine_baseline, engine_resnet], ["BaselineCNN", "ResNetStyle"])

## 7.7 综合评估：测试集表现

In [ ]:
def test_report(engine, name):
    test_loss, test_acc, test_preds, test_labels = engine.evaluate(test_loader, desc=f"Testing {name}")
    print(f"\n{'='*60}")
    print(f"  {name} - Test Results")
    print(f"{'='*60}")
    print(f"  Loss:     {test_loss:.4f}")
    print(f"  Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"{'='*60}")
    return test_preds, test_labels

preds_baseline, labels_baseline = test_report(engine_baseline, "BaselineCNN")
preds_resnet, labels_resnet = test_report(engine_resnet, "ResNetStyle")

## 7.8 混淆矩阵与各类准确率

In [ ]:
def plot_confusion_matrix(preds, labels, title="Confusion Matrix"):
    cm = confusion_matrix(labels.numpy(), preds.numpy())
    cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CIFAR10_CLASSES,
                yticklabels=CIFAR10_CLASSES, ax=ax1, cbar_kws={"label": "Count"})
    ax1.set_title(f"{title}\nConfusion Matrix (Counts)")
    ax1.set_ylabel("True Label")
    ax1.set_xlabel("Predicted Label")

    sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="YlOrRd",
                xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES,
                ax=ax2, vmin=0, vmax=1, cbar_kws={"label": "Proportion"})
    ax2.set_title(f"{title}\nConfusion Matrix (Normalized)")
    ax2.set_ylabel("True Label")
    ax2.set_xlabel("Predicted Label")

    plt.tight_layout()
    plt.show()

plot_confusion_matrix(preds_resnet, labels_resnet, "ResNetStyle")

In [ ]:
def plot_per_class_accuracy(preds, labels, title="Per-Class Accuracy"):
    cm = confusion_matrix(labels.numpy(), preds.numpy())
    per_class_acc = cm.diagonal() / cm.sum(axis=1)

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ["#2ecc71" if acc >= np.mean(per_class_acc) else "#e74c3c" for acc in per_class_acc]
    bars = ax.bar(CIFAR10_CLASSES, per_class_acc, color=colors, edgecolor="black", linewidth=0.5)
    ax.axhline(y=np.mean(per_class_acc), color="blue", linestyle="--",
               label=f"Mean: {np.mean(per_class_acc):.2%}")

    for bar, acc in zip(bars, per_class_acc):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{acc:.2%}", ha="center", fontsize=10, fontweight="bold")

    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{title}\nMean Accuracy: {np.mean(per_class_acc):.2%}")
    ax.legend()
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

    print("\nPer-Class Accuracy:")
    for cls, acc in zip(CIFAR10_CLASSES, per_class_acc):
        print(f"  {cls:12s}: {acc:.2%}")

plot_per_class_accuracy(preds_resnet, labels_resnet, "ResNetStyle")

In [ ]:
print("\nClassification Report (ResNetStyle):")
print(classification_report(labels_resnet.numpy(), preds_resnet.numpy(),
                            target_names=CIFAR10_CLASSES, digits=4))

## 7.9 错误分析

找出模型最容易混淆的类别对。

In [ ]:
def show_misclassified(preds, labels, dataset, max_show=10):
    incorrect_mask = preds != labels
    incorrect_indices = np.where(incorrect_mask.numpy())[0]

    if len(incorrect_indices) == 0:
        print("No errors found!")
        return

    np.random.shuffle(incorrect_indices)
    n_show = min(max_show, len(incorrect_indices))

    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for i, ax in enumerate(axes.flat):
        if i >= n_show:
            ax.axis("off")
            continue
        idx = incorrect_indices[i]
        img, true_label = dataset[idx]
        if isinstance(img, torch.Tensor):
            img = img.permute(1, 2, 0).cpu().numpy()
            for c in range(3):
                img[:, :, c] = img[:, :, c] * CIFAR10_STD[c] + CIFAR10_MEAN[c]
            img = np.clip(img, 0, 1)
        pred_label = preds[idx].item()
        ax.imshow(img)
        ax.set_title(f"True: {CIFAR10_CLASSES[true_label]}\nPred: {CIFAR10_CLASSES[pred_label]}",
                     fontsize=9, color="red")
        ax.axis("off")
    plt.suptitle("Misclassified Examples", fontsize=13, color="red")
    plt.tight_layout()
    plt.show()

raw_test = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=None)
show_misclassified(preds_resnet, labels_resnet, raw_test)

## 7.10 超参数对比实验

快速对比不同优化器和学习率的配置（跑少量 epoch）。

In [ ]:
def quick_experiment(model_cls, optimizer_name, lr, epochs=10):
    model = model_cls().to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    elif optimizer_name == "Adam":
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    elif optimizer_name == "AdamW":
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            opt.zero_grad()
            loss = criterion(model(data), target)
            loss.backward()
            opt.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += data.size(0)
    return correct / total

configs = [
    ("SGD", 0.01),
    ("SGD", 0.1),
    ("Adam", 0.001),
    ("Adam", 0.01),
    ("AdamW", 0.001),
    ("AdamW", 0.01),
]

print("Hyperparameter Quick Comparison (10 epochs each)...")
print("-" * 50)
results = {}
for opt_name, lr in configs:
    acc = quick_experiment(ResNetStyle, opt_name, lr)
    results[(opt_name, lr)] = acc
    print(f"  {opt_name:6s} lr={lr:<8.4f} -> Val Acc: {acc:.4f}")

best_config = max(results, key=results.get)
print(f"\nBest: {best_config[0]} lr={best_config[1]} -> {results[best_config]:.4f}")

## 7.11 模型导出与推理示例

In [ ]:
export_path = CHECKPOINT_DIR / "cifar10_final.pt"

engine_resnet.save(str(export_path))

export_model = ResNetStyle()
checkpoint = torch.load(export_path, map_location="cpu", weights_only=False)
export_model.load_state_dict(checkpoint["model_state_dict"])
export_model.eval()

scripted = torch.jit.script(export_model.cpu())
jit_path = CHECKPOINT_DIR / "cifar10_model_scripted.pt"
scripted.save(str(jit_path))
print(f"TorchScript model saved to {jit_path}")
print(f"Size: {jit_path.stat().st_size / 1e6:.2f} MB")

In [ ]:
print("Inference Demo:")
demo_img, demo_label = test_dataset[0]
demo_batch = demo_img.unsqueeze(0)

with torch.no_grad():
    logits = export_model(demo_batch)
    probs = F.softmax(logits, dim=1)
    pred_class = logits.argmax(dim=1).item()

print(f"  True label: {CIFAR10_CLASSES[demo_label]}")
print(f"  Predicted:  {CIFAR10_CLASSES[pred_class]}")
print(f"  Confidence: {probs[0, pred_class].item():.2%}")
print(f"\n  Top-5 predictions:")
top5_probs, top5_indices = torch.topk(probs, 5, dim=1)
for i in range(5):
    cls = CIFAR10_CLASSES[top5_indices[0, i].item()]
    prob = top5_probs[0, i].item()
    marker = " <<<" if top5_indices[0, i].item() == demo_label else ""
    print(f"    {i+1}. {cls:12s}: {prob:.2%}{marker}")

## 7.12 模型集成

多个模型的预测取平均，通常能提升 1-3% 准确率。

In [ ]:
def ensemble_predict(models, loader):
    for model in models:
        model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for data, target in tqdm(loader, desc="Ensemble"):
            data = data.to(device)
            ensemble_logits = None
            for model in models:
                logits = model(data)
                if ensemble_logits is None:
                    ensemble_logits = F.softmax(logits, dim=1)
                else:
                    ensemble_logits += F.softmax(logits, dim=1)
            ensemble_logits /= len(models)
            preds = ensemble_logits.argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(target)
    return torch.cat(all_preds), torch.cat(all_labels)

model1 = ResNetStyle()
ckpt1 = torch.load(CHECKPOINT_DIR / "resnet_style.pt", map_location=device, weights_only=False)
model1.load_state_dict(ckpt1["model_state_dict"])
model1.to(device)

model2 = BaselineCNN()
ckpt2 = torch.load(CHECKPOINT_DIR / "baseline_cnn.pt", map_location=device, weights_only=False)
model2.load_state_dict(ckpt2["model_state_dict"])
model2.to(device)

ensemble_preds, ensemble_labels = ensemble_predict([model1, model2], test_loader)
ensemble_acc = (ensemble_preds == ensemble_labels).float().mean().item()
print(f"\nEnsemble Accuracy: {ensemble_acc:.4f} ({ensemble_acc*100:.2f}%)")

## 7.13 项目总结

### 技术总结

| 项目阶段 | 使用的技术 |
|---------|-----------|
| **数据处理** | Dataset, DataLoader, Transforms, 数据增强 |
| **模型架构** | CNN, BatchNorm, 残差连接, Dropout, 全局平均池化 |
| **训练技巧** | SGD/AdamW, CosineAnnealingLR, 权重衰减, 梯度更新 |
| **评估方法** | 混淆矩阵, 各类准确率, 错误分析 |
| **工程实践** | Checkpoint 保存/加载, 模型导出, TorchScript |

### 效果对比

| 模型 | 参数量 | CIFAR-10 准确率 |
|------|--------|----------------|
| 基线 CNN | ~300K | ~75-80% |
| ResNet 风格 | ~600K | ~88-92% |
| 迁移学习 ResNet-18 | ~11M | ~90-93% |
| 集成模型 | ~900K | ~90-93% |

### 关键收获

1. **BatchNorm + 残差连接**是提升深度网络性能的标配
2. **数据增强**在小数据集上至关重要
3. **学习率调度**比固定的学习率效果好得多
4. **迁移学习**在大多数场景下是最优选择
5. **错误分析**帮助理解模型的短板

### 延伸方向
- 尝试更现代架构：ConvNeXt, ViT (Vision Transformer)
- 使用 MixUp / CutMix 等高级数据增强
- 在自定义数据集上实践完整的 CNN 流水线
- 部署模型到移动端 (ONNX / CoreML)